In [1]:
# ============================================================
# 1 — Imports & Configuration
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

PROJECT_ROOT = Path.cwd().parent

TRAIN_PATH = PROJECT_ROOT / "Data" / "processed" / "project1_train.csv"
VAL_PATH   = PROJECT_ROOT / "Data" / "processed" / "project1_validation.csv"
TEST_PATH  = PROJECT_ROOT / "Data" / "processed" / "project1_test.csv"

train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VAL_PATH)
test = pd.read_csv(TEST_PATH)

TARGET = "Product"
TEXT_COL = "Consumer complaint narrative"

X_train = train[TEXT_COL].fillna("")
y_train = train[TARGET]

X_val = validation[TEXT_COL].fillna("")
y_val = validation[TARGET]

X_test = test[TEXT_COL].fillna("")
y_test = test[TARGET]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (58422,)
Validation: (11685,)
Test: (11685,)


In [3]:
# ============================================================
# CELL 2 — Evaluation Function
# ============================================================

def evaluate_model(model, X_val, y_val, model_name):

    predictions = model.predict(X_val)

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Macro F1": f1_score(
            y_val,
            predictions,
            average="macro"
        ),
        "Weighted F1": f1_score(
            y_val,
            predictions,
            average="weighted"
        )
    }

    print(f"\n{'=' * 70}")
    print(model_name)
    print(f"{'=' * 70}")

    print(f"Accuracy:    {results['Accuracy']:.4f}")
    print(f"Macro F1:    {results['Macro F1']:.4f}")
    print(f"Weighted F1: {results['Weighted F1']:.4f}")

    return results, predictions

### Experiment 1 — reproduce benchmark

In [4]:
# ============================================================
# CELL 3 — Experiment 1
# Word TF-IDF (1,2) + Logistic Regression
# ============================================================

word_tfidf_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            solver="lbfgs"
        )
    )
])

print("Training Experiment 1...")

word_tfidf_lr.fit(
    X_train,
    y_train
)

exp1_results, exp1_predictions = evaluate_model(
    word_tfidf_lr,
    X_val,
    y_val,
    "Word TF-IDF (1,2) + Logistic Regression"
)

Training Experiment 1...

Word TF-IDF (1,2) + Logistic Regression
Accuracy:    0.8340
Macro F1:    0.7144
Weighted F1: 0.8264


### Experiment 2 — character representation

In [5]:
# ============================================================
# CELL 4 — Experiment 2
# Character TF-IDF + Logistic Regression
# ============================================================

char_tfidf_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=3,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            solver="lbfgs"
        )
    )
])

print("Training Experiment 2...")

char_tfidf_lr.fit(
    X_train,
    y_train
)

exp2_results, exp2_predictions = evaluate_model(
    char_tfidf_lr,
    X_val,
    y_val,
    "Character TF-IDF (3,5) + Logistic Regression"
)

Training Experiment 2...

Character TF-IDF (3,5) + Logistic Regression
Accuracy:    0.8385
Macro F1:    0.7330
Weighted F1: 0.8327


### Experiment 3 — Word + Character

In [6]:
# ============================================================
# CELL 5 — Experiment 3
# Word + Character TF-IDF
# ============================================================

combined_tfidf_lr = Pipeline([
    (
        "features",
        FeatureUnion([
            (
                "word",
                TfidfVectorizer(
                    lowercase=True,
                    strip_accents="unicode",
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    sublinear_tf=True
                )
            ),
            (
                "char",
                TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(3, 5),
                    min_df=3,
                    sublinear_tf=True
                )
            )
        ])
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            solver="lbfgs"
        )
    )
])

print("Training Experiment 3...")

combined_tfidf_lr.fit(
    X_train,
    y_train
)

exp3_results, exp3_predictions = evaluate_model(
    combined_tfidf_lr,
    X_val,
    y_val,
    "Word + Character TF-IDF + Logistic Regression"
)

Training Experiment 3...

Word + Character TF-IDF + Logistic Regression
Accuracy:    0.8442
Macro F1:    0.7461
Weighted F1: 0.8391


### Comparision

In [7]:
# ============================================================
# CELL 6 — Representation Comparison
# ============================================================

results = pd.DataFrame([
    exp1_results,
    exp2_results,
    exp3_results
])

results.sort_values(
    "Macro F1",
    ascending=False
).round(4)

,Model,Accuracy,Macro F1,Weighted F1
2,Word + Character TF-IDF + Logistic Regression,0.8442,0.7461,0.8391
1,"Character TF-IDF (3,5) + Logistic Regression",0.8385,0.7330,0.8327
0,"Word TF-IDF (1,2) + Logistic Regression",0.8340,0.7144,0.8264


### Analyze The Winner

In [8]:
# ============================================================
# CELL 7 — Improvement Over Benchmark
# ============================================================

benchmark_macro_f1 = exp1_results["Macro F1"]

results["Macro F1 Improvement"] = (
    results["Macro F1"] - benchmark_macro_f1
)

results["Macro F1 Improvement %"] = (
    results["Macro F1 Improvement"]
    / benchmark_macro_f1
    * 100
)

results.sort_values(
    "Macro F1",
    ascending=False
).round(4)

,Model,Accuracy,Macro F1,Weighted F1,Macro F1 Improvement,Macro F1 Improvement %
2,Word + Character TF-IDF + Logistic Regression,0.8442,0.7461,0.8391,0.0317,4.4414
1,"Character TF-IDF (3,5) + Logistic Regression",0.8385,0.7330,0.8327,0.0187,2.6125
0,"Word TF-IDF (1,2) + Logistic Regression",0.8340,0.7144,0.8264,0.0000,0.0000


### Per-Class comparision

In [9]:
# ============================================================
# CELL 8 — Per-Class Performance of Best Representation
# ============================================================

best_row = results.loc[
    results["Macro F1"].idxmax()
]

print("Best representation:")
print(best_row["Model"])

if best_row["Model"].startswith("Word + Character"):
    best_predictions = exp3_predictions
elif best_row["Model"].startswith("Character"):
    best_predictions = exp2_predictions
else:
    best_predictions = exp1_predictions

print(
    classification_report(
        y_val,
        best_predictions,
        digits=4
    )
)

Best representation:
Word + Character TF-IDF + Logistic Regression
                                                         precision    recall  f1-score   support

                            Checking or savings account     0.7697    0.8707    0.8171      2096
                                            Credit card     0.8123    0.8437    0.8277      1970
    Credit reporting or other personal consumer reports     0.8544    0.7379    0.7919       477
                                        Debt collection     0.8951    0.9472    0.9205      3848
                              Debt or credit management     1.0000    0.2524    0.4031       103
     Money transfer, virtual currency, or money service     0.7641    0.6374    0.6950       935
                                               Mortgage     0.9428    0.9304    0.9366       762
Payday loan, title loan, personal loan, or advance loan     0.7147    0.5534    0.6238       412
                                           Prepaid card    

### Final Decision 

In [10]:
# ============================================================
# CELL 9 — Decision Gate
# ============================================================

best_model_name = results.loc[
    results["Macro F1"].idxmax(),
    "Model"
]

best_macro_f1 = results["Macro F1"].max()

print(f"""
PROJECT 1 — FEATURE REPRESENTATION DECISION
============================================

Benchmark:
    Word TF-IDF (1,2) + Logistic Regression

Benchmark Macro F1:
    {benchmark_macro_f1:.4f}

Best representation:
    {best_model_name}

Best Macro F1:
    {best_macro_f1:.4f}

Decision:
    Use the strongest representation as the candidate
    model for the next experiment.

Next:
    → Controlled metadata experiment
    → Then hyperparameter tuning
    → Then final error analysis
    → Then untouched test evaluation
""")


PROJECT 1 — FEATURE REPRESENTATION DECISION

Benchmark:
    Word TF-IDF (1,2) + Logistic Regression

Benchmark Macro F1:
    0.7144

Best representation:
    Word + Character TF-IDF + Logistic Regression

Best Macro F1:
    0.7461

Decision:
    Use the strongest representation as the candidate
    model for the next experiment.

Next:
    → Controlled metadata experiment
    → Then hyperparameter tuning
    → Then final error analysis
    → Then untouched test evaluation

